<a href="https://colab.research.google.com/github/gloriabee/DEMovies/blob/main/gold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import os
import duckdb

DRIVE_DB= "/content/drive/MyDrive/data_engineering/movies.duckdb"



In [ ]:
print("Exists:",os.path.exists(DRIVE_DB))


if os.path.exists(DRIVE_DB):
  print("Drive DB size:",os.path.getsize(DRIVE_DB))

In [ ]:
con=duckdb.connect(DRIVE_DB)
con.execute("SHOW TABLES").fetchdf()

In [ ]:
movies_df=con.execute("""
SELECT *
FROM movies_silver
""").fetchdf()

In [ ]:
movies_df.head()

In [ ]:
movies_writers_df=con.execute("""
SELECT *
FROM movies_writers_silver
""").fetchdf()
movies_writers_df.head()

In [ ]:
movies_directors_df=con.execute("""
SELECT *
FROM movies_directors_silver
""").fetchdf()
movies_directors_df.head()

In [ ]:
movies_genres_df=con.execute("""
SELECT *
FROM movies_genres_silver
""").fetchdf()
movies_genres_df.head()

### Gold Layer

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_movie AS
SELECT
  ROW_NUMBER() OVER (ORDER BY tconst) AS movie_key,
  tconst,
  primaryTitle,
  startYear,
  runtimeMinutes
FROM movies_silver
""")

In [ ]:
con.execute("""
select * from dim_movie
""").fetchdf()

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_genre AS
SELECT
    ROW_NUMBER() OVER (ORDER BY genre) AS genre_key,
    genre
FROM (
    SELECT DISTINCT genre
    FROM movies_genres_silver
)
""")

In [ ]:
con.execute("""
select * from dim_genre
""").fetchdf()

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE movie_genre AS
SELECT
    m.movie_key,
    g.genre_key
FROM movies_genres_silver mg
JOIN dim_movie m
    ON mg.tconst = m.tconst
JOIN dim_genre g
    ON mg.genre = g.genre
""")

In [ ]:
con.execute("""
select * from movie_genre
""").fetchdf()

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_director AS
SELECT
    ROW_NUMBER() OVER (ORDER BY director) AS director_key,
    director
FROM (
    SELECT DISTINCT director
    FROM movies_directors_silver
)
""")

In [ ]:
con.execute("""
select * from dim_director
""").fetchdf()

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE bridge_movie_director AS
SELECT
    m.movie_key,
    d.director_key
FROM movies_directors_silver md
JOIN dim_movie m
    ON md.tconst = m.tconst
JOIN dim_director d
    ON md.director = d.director
""")

In [ ]:
con.execute("""
select * from bridge_movie_director
""").fetchdf()

In [31]:
con.execute("""
CREATE OR REPLACE TABLE dim_writer AS
SELECT
    ROW_NUMBER() OVER (ORDER BY writer) AS writer_key,
    writer
FROM (
    SELECT DISTINCT writer
    FROM movies_writers_silver
)
""")

In [32]:
con.execute("""
select * from dim_writer
""").fetchdf()

,writer_key,writer
0,1,'Weird Al' Yankovic
1,2,A.A. Milne
2,3,A.B. Guthrie Jr.
3,4,A.B. Rahman El-Sharkawi
4,5,A.E. Southon
...,...,...
7169,7170,Éric Besnard
7170,7171,Éric Lartigau
7171,7172,Éric Toledano
7172,7173,Ömer Vargı


In [33]:
con.execute("""
CREATE OR REPLACE TABLE bridge_movie_writer AS
SELECT
    m.movie_key,
    w.writer_key
FROM movies_writers_silver mw
JOIN dim_movie m
    ON mw.tconst = m.tconst
JOIN dim_writer w
    ON mw.writer = w.writer
""")

In [34]:
con.execute("""
select * from bridge_movie_writer
""").fetchdf()

,movie_key,writer_key
0,1290,6303
1,1290,2149
2,415,4344
3,415,2131
4,2528,3557
...,...,...
12765,4573,3709
12766,538,2585
12767,538,1589
12768,538,6263


### Fact table

In [35]:
con.execute("""
CREATE OR REPLACE TABLE fact_movie_performance AS
SELECT
    m.movie_key,
    ms.averageRating,
    ms.numVotes,
    ms.rank,
    ms.runtimeMinutes
FROM movies_silver ms
JOIN dim_movie m
    ON ms.tconst = m.tconst
""")

In [36]:
con.execute("""
select * from fact_movie_performance
""").fetchdf()

,movie_key,averageRating,numVotes,rank,runtimeMinutes
0,1290,9.3,3246333,1,142
1,415,9.2,2261044,2,175
2,2528,9.1,3233997,3,152
3,1727,9.0,2197501,4,201
4,1215,9.0,1606270,5,196
...,...,...,...,...,...
4995,4854,6.3,41407,4996,100
4996,4039,6.2,41310,4997,150
4997,3365,6.3,41132,4998,87
4998,4573,6.2,41131,4999,102


### Genre Level Gold Aggregation

In [38]:
con.execute("""
CREATE OR REPLACE TABLE gold_genre_performance AS
SELECT
    g.genre,
    COUNT(DISTINCT f.movie_key) AS movie_count,
    ROUND(AVG(f.averageRating), 2) AS avg_rating,
    ROUND(AVG(f.runtimeMinutes), 2) AS avg_runtime,
    SUM(f.numVotes) AS total_votes
FROM fact_movie_performance f
JOIN movie_genre bg
    ON f.movie_key = bg.movie_key
JOIN dim_genre g
    ON bg.genre_key = g.genre_key
GROUP BY g.genre
""")

In [39]:
con.execute("""
select * from gold_genre_performance
""").fetchdf()

,genre,movie_count,avg_rating,avg_runtime,total_votes
0,Adventure,913,7.18,115.58,245712396.0
1,History,232,7.32,131.77,26715946.0
2,Western,58,7.46,124.22,9111137.0
3,Music,152,7.24,114.95,16321710.0
4,Film-Noir,29,7.76,100.38,2195906.0
5,Documentary,86,7.76,99.57,4305428.0
6,News,2,7.95,95.00,82787.0
7,Drama,3213,7.26,119.21,463398037.0
8,Crime,1117,7.17,116.59,179489749.0
9,Thriller,792,7.09,114.84,148070658.0


### Year-level Gold aggregation

In [40]:
con.execute("""
CREATE OR REPLACE TABLE gold_yearly_movie_summary AS
SELECT
    m.startYear,
    COUNT(*) AS movie_count,
    ROUND(AVG(f.averageRating), 2) AS avg_rating,
    ROUND(AVG(f.runtimeMinutes), 2) AS avg_runtime,
    SUM(f.numVotes) AS total_votes
FROM fact_movie_performance f
JOIN dim_movie m
    ON f.movie_key = m.movie_key
GROUP BY m.startYear
ORDER BY m.startYear
""")

In [41]:
con.execute("""
select * from gold_yearly_movie_summary
""").fetchdf()

,startYear,movie_count,avg_rating,avg_runtime,total_votes
0,1920,1,8.00,67.00,77099.0
1,1921,1,8.20,68.00,147342.0
2,1922,1,7.80,95.00,124574.0
3,1923,1,8.10,74.00,25213.0
4,1924,1,8.10,45.00,67190.0
...,...,...,...,...,...
102,2022,135,7.10,123.14,17912944.0
103,2023,127,7.06,122.78,15263749.0
104,2024,105,6.99,124.04,11435215.0
105,2025,97,7.00,122.40,11281158.0
